In [395]:
from graphviz import Digraph

def trace(root):
    """
    builds a set of all nodes and edges in a graph
    """
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
    nodes, edges = trace(root)    
    for n in nodes:
        uid = str(id(n))
        # for any value in the graph,create a rectangular ('record') node for it
        dot.node(name = uid, label = "{%s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name = uid + n._op, label=n._op)
            # and connect this node to it
            dot.edge(uid + n._op, uid)
    
    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)
    
    return dot

In [396]:
import torch
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [397]:
x1 = torch.tensor([2.0],  dtype=torch.double, requires_grad=True)
x2 = torch.tensor([0.0],  dtype=torch.double, requires_grad=True)
w1 = torch.tensor([-3.0], dtype=torch.double, requires_grad=True)
w2 = torch.tensor([1.0],  dtype=torch.double, requires_grad=True)
b  = torch.tensor([6.8813735870195432], dtype=torch.double, requires_grad=True)

In [398]:
n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

print(f"o: {o.item()}")
o.backward()
print("=============")
print('x2:', x2.grad.item())
print('w2:', w2.grad.item())
print('x1:', x1.grad.item())
print('w1:', w1.grad.item())

o: 0.7071067811865476
x2: 0.49999999999999994
w2: 0.0
x1: -1.4999999999999998
w1: 0.9999999999999999


In [399]:
o

tensor([0.7071], dtype=torch.float64, grad_fn=<TanhBackward0>)

In [400]:
x2.grad

tensor([0.5000], dtype=torch.float64)

In [401]:
class Value:
    """ stores a single scalar value and its gradient """

    def __init__(self, data, _children=(), _op='', label=''):
        self.data = data
        self.grad = 0.0
        # internal variables used for autograd graph construction
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op # the op that produced this node, for graphviz / debugging / etc
        self.label = label
        
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"
    
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward

        return out
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        
        return out
    
    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward

        return out
    
    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward

        return out
    
    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        
        return out
    
    def exp(self):
        x = self.data
        t = math.exp(x)
        out = Value(t, (self,), 'exp')
        
        def _backward():
            self.grad += t * out.grad
        out._backward = _backward
        
        return out
    
    def backward(self):
        
        # build the topological ordered list of all the children in the DAG
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
 
        # We must go through the topogical ordered list in the reverse order as per backpropagation
        # We must go through one variable at a time and
        # then apply the chain rule to get its gradient
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()
    
    def __neg__(self): # -self
        return self * -1

    def __radd__(self, other): # other + self
        return self + other

    def __sub__(self, other): # self - other
        return self + (-other)

    def __rsub__(self, other): # other - self
        return other + (-self)

    def __rmul__(self, other): # other * self
        return self * other

    def __truediv__(self, other): # self / other
        return self * other**-1

    def __rtruediv__(self, other): # other / self
        return other * self**-1

In [402]:
class Neuron:
    def __init__(self, nin):
        self.w = [Value(np.random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(np.random.uniform(-1, 1))
    
    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out
    
    def parameters(self):
        return self.w + [self.b]

In [403]:
x=[2.0, 3.0, -1.0]
n = Neuron(3)
n(x)

Value(data=0.5175451994726584, grad=0.0)

now to form a layer of neurons

In [404]:
class Layer:
    """ a layer of neurons in a MLP """
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

In [405]:
layer = Layer(3, 10)
layer(x)

[Value(data=0.9999012541940075, grad=0.0),
 Value(data=-0.4743170236398523, grad=0.0),
 Value(data=0.8641422569584488, grad=0.0),
 Value(data=0.9954342293003552, grad=0.0),
 Value(data=-0.9996122433952422, grad=0.0),
 Value(data=-0.5399294958300719, grad=0.0),
 Value(data=-0.7058064072101627, grad=0.0),
 Value(data=-0.5532039138689823, grad=0.0),
 Value(data=-0.9972421426764835, grad=0.0),
 Value(data=-0.9975815011480257, grad=0.0)]

In [406]:
class MLP:
    """ a multi-layer perceptron """
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [407]:
mlp1 = MLP(3, [4, 4, 1])
mlp1(x)

Value(data=0.3084996132403804, grad=0.0)

In [408]:
#draw_dot(mlp1(x)).render('mlp1', view=True)

In [409]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0, -1.0, -1.0, 1.0] #desired outputs for the above inputs

In [410]:
ypred = [mlp1(x) for x in xs]
ypred

[Value(data=0.3084996132403804, grad=0.0),
 Value(data=0.12326294897375989, grad=0.0),
 Value(data=0.003149122757099115, grad=0.0),
 Value(data=0.5567171619645246, grad=0.0)]

In [411]:
for epoch in range(2000):
    #forward pass
    ypred = [mlp1(x) for x in xs]
    loss = sum((yout - ygt)**2 for yout, ygt in zip(ypred, ys))

    #zero the gradients before running the backward pass
    for p in mlp1.parameters():
        p.grad = 0.0
    
    # backpropagate to get the gradients
    loss.backward()
    
    # update the weights using gradient descent
    for p in mlp1.parameters():
        p.data -= 0.05 * p.grad

    print(f"Epoch {epoch}: loss = {loss.data}")

#ypred = [mlp1(x) for x in xs]
#loss = sum((yout - ygt)**2 for yout, ygt in zip(ypred, ys))
#loss

Epoch 0: loss = 2.942700274411055
Epoch 1: loss = 1.4356099168896939
Epoch 2: loss = 0.6912939298262893
Epoch 3: loss = 0.360346172166892
Epoch 4: loss = 0.2552623998001292
Epoch 5: loss = 0.19716317081100576
Epoch 6: loss = 0.16011044271779118
Epoch 7: loss = 0.13448069011565678
Epoch 8: loss = 0.11572527067731045
Epoch 9: loss = 0.10142119563030383
Epoch 10: loss = 0.09016167209447978
Epoch 11: loss = 0.08107478132862093
Epoch 12: loss = 0.07359170277988245
Epoch 13: loss = 0.06732571615791962
Epoch 14: loss = 0.06200482596713454
Epoch 15: loss = 0.05743221304655538
Epoch 16: loss = 0.05346197747556389
Epoch 17: loss = 0.049983696917073314
Epoch 18: loss = 0.04691227837201301
Epoch 19: loss = 0.04418110190679281
Epoch 20: loss = 0.04173727507340792
Epoch 21: loss = 0.03953827736534057
Epoch 22: loss = 0.037549542070260705
Epoch 23: loss = 0.03574268377748623
Epoch 24: loss = 0.034094179109359815
Epoch 25: loss = 0.03258437108809555
Epoch 26: loss = 0.031196708216320885
Epoch 27: loss

In [412]:
ypred

[Value(data=0.9930746997990515, grad=-0.013850600401897006),
 Value(data=-0.9918885852457862, grad=0.016222829508427683),
 Value(data=-0.9906310271471118, grad=0.01873794570577636),
 Value(data=0.9917906917070204, grad=-0.016418616585959223)]